## SVM example using a 2D dataset.

The majority of this notebook file is adopted from the course materials of **CMSE 202 Introduction to Computational Modeling and Data Analysis II, Department of Computational Mathematics, Science and Engineering, Michigan State University.**


---
<a id="svm"></a>
## 1. The SVM Classifier

* The algorithm creates a line and marks the points from each class that are the "closest" to the line. These points are called the **support vectors**. In the image at the top, the black dots have one support vector and the white circles have two (two points equidistant from the line). We then compute the distance between the line and the support vectors. This distance is called the **margin**. 

* An SVM tries to make a decision boundary, in the form of a line, such that the separation between the two classes is as wide as possible.

In this class, we'll try running an SVM classifier and then see if we can judge how well it did. We can compare this to the line you made by hand in the pre-class assignment.

### 1.1 Classify your "blob" data

In [ ]:
# imports for the day
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn import svm

In [ ]:
X,y = make_blobs(n_features=2, centers=2, random_state=3)
clrs = ['red' if cls == 1 else 'green' for cls in y]
plt.scatter(X[:,0], X[:,1], c=clrs)
xline = np.linspace(-6,3)
yline = xline * -2 -1.5
plt.plot(xline,yline);

print(y)

### 1.2 Running the SVM classifier

The basic process for making an SVM classifier, as with all classifiers, is a follows:

- create the model
- fit the model
- examine the result

Rather than trying to build an SVM classifier from scratch like you did with the Perceptron model, we're going to use the `sklearn` module. You can find the main scikit-learn documentation [here](https://scikit-learn.org/stable/user_guide.html).

To accomplish the steps above, you would use the `sklearn.svm.svc` (short for "Support Vector Classifier") to do the work. Here are some of the details.
- To build the model you would use `svm.SVC`. This constructor takes a number of parameters:
   - you can see all the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html">parameters here</a>.
   - the most important are the first two:
      - **C** this is a parameter of how the fit is to be done. We will explore this later, for now the default is fine.
      - **kernel** for linear separability we are going to use the `"linear"` kernel. Note that the kernel name is a string.
   - The constructor also returns a model
- You then call `my_model.fit()` (or whatever you called your model variable) to fit the data. It takes two arguments:
   - The array of data. Its shape is (`n_samples`, `n_features`). That is, the provided array is `n_samples` large and each sample (each element in the array) is an array of `n_features`.
   - An array of class labels. It should be the size of `n_samples`, one for each data element. Typically these are integers.
- This function returns the same model you called `fit()` from (thus you don't need the return value)

**Note: we are not using `train_test_split` for this model, you will fit your SVM model to the whole data set. We are simply comparing the linear separation you guessed to the one SVC produces.**

&#9989; **Do This:** Create an SVC classifier using the `"linear"` kernel and with `C=10` (if you don't specify the value for `C` argument, it will use the default, which is `1.0`). Then, fit the model to your data. **Note**: you should not expect any meaningful output just yet, so as long as your call to `fit()` doesn't throw an error, you should be good to move on.

In [ ]:
# build classifier and call the fit function on your data here

cls = svm.SVC(kernel="linear", C=10)
print(X.shape, y.shape)
cls.fit(X,y)




### 1.3 Upacking the SVM results

What information can we get from the fitted SVM? Remember that these fitted models have [a number of attributes and methods](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) that provide use with information regarding the fit.

There are some useful attributes:

- The attributes `coef_` and `intercept_` (note the underline) can be used to draw the the boundary the classifier created. We'll talk more about that below.
- The attribute `support_vectors_` are the coordinates of the support vectors used by the algorithm. In this case these are the 2D location of the data points that were determined to be the support vectors.

And some useful methods:

- `predict` takes single parameter, an array of shape (`number_of_data_to_predict`, `number_of_features`) and returns an array of class predictions.
- `decision_function` takes single parameter, an array of shape (`number_of_data_to_predict`, `number_of_features`) and returns the distance from the decision border for each datum. The sign indicates the class the datum belongs to.
- `score`, takes two parameters: array sample data of shape (`number_of_data_to_predict`, `number_of_features`) and array of size `number_of_data_to_predict` class labels and returns a percentage correct.

&#9989; **Do This:**  Explore some of the methods above and write a bit of code that prints the accuracy of our formed classifier. What function should you use for this? Check the accuracy using the same data you used for find the model fit -- does the result match your expectations?

In [ ]:
print(f"Accuracy: {cls.score(X,y)}")

---
<a id="viz"></a>
## 2. Visualizing the boundary and the margins

We won't go through the nitty-gritty details for "how" an SVM algorithm defines the optimal boundary -- the pre-class video described conceptually what is happening -- but we are going to try and get a few terms straight so we can plot some results and compare to our prediction.

### 2.1 What is a hyperplane?

A <a href="https://en.wikipedia.org/wiki/Hyperplane"> hyperplane </a> is geometric element whose dimensionality is one less than the space in which it is embedded. For our purposes, a hyperplane divides the embedded space. Thus:
- a 2D space can be divided by a 1D line
- a 3D space can be divided by a 2D plane
- and so on...
   
The data that we generated is in 2D and thus we are trying to find a 1D hyperplane -- a line -- to divide the space. In general, an SVM can search in any n-dimensional space and find a n-1 hyperplane (or at least find the best one it is capable of) that divides the space. SVM literature thus talks always about hyperplanes as a more general concept. For the moment we will be content with hyperplanes in 1D, **which are lines**.

### 2.2 Drawing the boundary

We are going to use the attributes of the fitted SVM to draw the line that was determined by the mode. First, let's remember that the equation of a line in slope-intercept format is:

$$ y = mx + b $$

Where $m$ is the slope and $b$ is the intercept. However, there is another form of a line equation that is called the "standard" form which is:

$$ Ax + By + C = 0 $$

where $A$ and $B$ are the **coefficients** of the line. You should be able to convert back and forth between these two forms relatively easily. We had to do this when thinking about how to draw the best fit line from the Perceptron model as well!

These standard coefficients are what are returned by the `.coef_` attributes of the classifier hyperplane. Assuming that we are doing a 2D (2 feature) classifier, the coefficients represent a line in standard format. Thus:

$$ Ax + By + C = 0 $$

then `.coef_` returns an array of $[A, B]$ and `intercept_` returns the $C$ value of the line.

By pulling that information from the model, you should be able to determine the slope-intercept form and draw the decision boundary through the "blobs". 

&#9989; **Do This:** Plot the blobs again along with your predicted line from the pre-class assignment **as well as the line from the model (i.e., the decision boundary)**.

In [ ]:
# Create the hyperplane
w = cls.coef_[0]
slope = -w[0] / w[1]
intercept = (cls.intercept_[0]) / w[1]
xx = np.linspace(-6, 2.5)
yy = slope * xx - intercept

# Plot the hyperplane
plt.plot(xx, yy, 'r--')

## Plot the guess we made
xline = np.linspace(-6,3)
yline = xline * -2 -1.5
plt.plot(xline,yline, 'b-')

plt.legend(['Decision Boundary', 'Our Guess'])


plt.scatter(X[:,0], X[:,1], c=clrs)

print(f"Slope:{slope:.3}, Intercept:{intercept:.3}")

### 2.3 Draw the Support Vectors
Remember that the attribute `support_vectors_` (note the trailing underline) of the model yields a list of points that are used as support vectors. 

&#9989; **Do This:** Using `support_vectors_`, redraw the blobs and boundary but now also mark the support vectors the algorithm used to determine the boundary. If you have trouble-coming up with a good way of marking the support vectors, chat with your group to brainstorm ideas!

In [ ]:
# draw with support vectors

plt.plot(xx, yy)

plt.scatter(X[:,0], X[:,1], c=clrs)

sv = cls.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=200,
           linewidth=1, facecolors='none', edgecolors='k');

### 2.4 SVM Margins

Besides drawing the boundary, the SVM tries to draw a **margin**, a set of parallel lines on either side of the boundary. You learned about this in the pre-class video.

This width of this margin is controlled by the `C` parameter. When you create the model (the `C` parameter is always positive), `C` tells the SVM optimization how much you want to avoid misclassifying each training example. For large values of `C`, the optimization will choose a smaller-margin hyperplane if that hyperplane does a better job of getting all the training points classified correctly. Conversely, a very small value of `C` will cause the optimizer to look for a larger-margin separating hyperplane, even if that hyperplane misclassifies more points. For very tiny values of `C`, you should get misclassified examples, often even if your training data is linearly separable. 

* Large `C` - better job of classifying training data; smaller margin
* Smaller `C` - more misclassifcations; larger margin

#### 2.4.1 Calculating the margin

Below, we have written a function that can help you draw the margins so that you can see the effect that `C` has on the SVM results. To think about how this works, let's look at the image from the top of the notebook again:

<!--
<img src="https://i.stack.imgur.com/7sFL3.png" width=300px>
-->

<img src="https://i.ibb.co/Y73PXKYV/7sFL3.jpg" width=300px>

The way that margins are determined follows this procedure:

- Looking at the image, the distance between the two margins is always $\frac{2}{\| w \|}$, and the distance from the boundary to one margin half that value (the margins are equidistant from the boundary)
   - $w$ is the vector $[A,B]$ (the coefficients of x and y in standard form), so $\| w \| = \sqrt{A^2 + B^2} $ and the margin distance is therefore $\frac{1}{\sqrt{A^2 + B^2}} $
- Given the boundary line x and y arrays, the updated $y_{margin-down}$ array is: $ y - \sqrt{1 + m^2} $ and $y_{margin-up}$ array is: $y + \sqrt{1 + m^2} $

#### 2.4.2 Describing the function `margin_y_fn`

The function below, `margin_y_fn`, takes 3 arguments:
- the y array of values of the boundary line
- The `A` coefficient from the fit
- The `B` coefficient from the fit

It returns two values:
- an array of y values for the lower margin (given the same x values as the boundary line)
- an array of y values for the upper margin (also given the same x values as the boundary line).

The function appears below. Make sure to run it before trying the next task.

In [ ]:
def margin_y_fn(y, A, B):
    slope = -A/B
    margin_dist = 1/np.sqrt(A**2 + B**2)
    y_down = y - np.sqrt(1 + slope** 2) * margin_dist
    y_up = y + np.sqrt(1 +  slope** 2) * margin_dist
    return y_down, y_up

In [ ]:

yy_down, yy_up = margin_y_fn(yy, cls.coef_[0][0], cls.coef_[0][1])

plt.plot(xx, yy)
plt.scatter(X[:,0], X[:,1], c=clrs)
plt.scatter(sv[:, 0], sv[:, 1], s=200,
           linewidth=1, facecolors='none', edgecolors='k')
plt.plot(xx, yy_down, 'k--')
plt.plot(xx, yy_up, 'k-')

---
<a id="hyper-tuning"></a>
## 3. Hyperparameter tuning (the effect of `C`)

As we discussed the parameter `C` controls that amount of misclassification that is acceptable:

* Large `C` - better job of classifying training data; smaller margin
* Smaller `C` - more misclassifcations; larger margin

In this last section, we will look into the effect of choosing a different value of `C` for the same data. We will pull together all the work you have done so far into a single cell. 

&#9989; **Do This:** Below, copy and paste all the necesssary code that you wrote above such that the cell below does the following:

* make your fake data
* create an SVC model and fit it to your data
* calculate the decision boundaries
* plot the data, the optimal line, the decision boundaries and the support vectors, and
* print the accuracy.

Make sure you run the cell and confirm that it produces the output you expect.

In [ ]:
# Create the data
X,y = make_blobs(n_features=2, centers=2, random_state=4)
clrs = ['red' if cls == 1 else 'green' for cls in y]

# Plot the data
plt.scatter(X[:,0], X[:,1], c=clrs)

# Create the fit
cls = svm.SVC(kernel="linear", C=10)
cls.fit(X,y)

# Create the hyperplane
w = cls.coef_[0]
slope = -w[0] / w[1]
intercept = (cls.intercept_[0]) / w[1]
xx = np.linspace(X[:,0].min(), X[:,0].max())
yy = slope * xx - intercept

# Plot the hyperplane
plt.plot(xx, yy, 'r--')

# Mark the support vectors
sv = cls.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=200,
           linewidth=1, facecolors='none', edgecolors='k')

# Plot the margins
yy_down, yy_up = margin_y_fn(yy, cls.coef_[0][0], cls.coef_[0][1])
plt.plot(xx, yy_down, 'k--')
plt.plot(xx, yy_up, 'k-')
plt.show()

print(f"Accuracy: {cls.score(X,y)}")

### 3.1 Automate the process to explore the effect of "C"

Now that you have pulled at that code together and checked that it works, put it inside a function called `test_C_parameter` that runs all that code and takes the following arguments:
* `C` - the hyperparamter that controls misclassifications
* `random_state` - the state associated with the created data

Then run the function using a variety of choices for `C` to observe how the margins change. You might need to change `random_state` to get distributions of blobs that overlap a bit more in order to really understand how changing `C` influences the results.

In [ ]:
def test_C_parameter(C=10, random_state=3):

    # Create the data
    X,y = make_blobs(n_features=2, centers=2, random_state=random_state)
    clrs = ['red' if cls == 1 else 'green' for cls in y]

    # Plot the data
    plt.scatter(X[:,0], X[:,1], c=clrs)

    # Create the fit
    cls = svm.SVC(kernel="linear", C=C)
    cls.fit(X,y)

    # Create the hyperplane
    w = cls.coef_[0]
    slope = -w[0] / w[1]
    intercept = (cls.intercept_[0]) / w[1]
    xx = np.linspace(X[:,0].min(), X[:,0].max())
    yy = slope * xx - intercept

    # Plot the hyperplane
    plt.plot(xx, yy, 'r--')

    # Mark the support vectors
    sv = cls.support_vectors_
    plt.scatter(sv[:, 0], sv[:, 1], s=200,
               linewidth=1, facecolors='none', edgecolors='k')

    # Plot the margins
    yy_down, yy_up = margin_y_fn(yy, cls.coef_[0][0], cls.coef_[0][1])
    plt.plot(xx, yy_down, 'k--')
    plt.plot(xx, yy_up, 'k-')
    plt.title('C = '+str(C))
    plt.show()
    
    print(f"Accuracy: {cls.score(X,y)}")
    
    return;

In [ ]:
test_C_parameter(C=10, random_state=4)
test_C_parameter(C=1, random_state=4)
test_C_parameter(C=0.1, random_state=4)
test_C_parameter(C=0.01, random_state=4)

---
### Example of SVM classifier using Iris data (鳶尾花). 

The three classes in the Iris dataset are represented by numbers in the 'target' vector:

    1. Iris-setosa (n=50)
    2. Iris-versicolor (n=50)
    3. Iris-virginica (n=50)

There are four measurements (features) for each flower in the  'data' matrix:    
    1. sepal length in cm
    2. sepal width in cm
    3. petal length in cm
    4. petal width in cm
    
Example:
![Image of flower](http://nbviewer.jupyter.org/github/rasbt/pattern_classification/blob/master/Images/iris_petal_sepal.png)

Most of the data and code are adopted from internet resources and ChatGPT. 

* Load Iris data.
* Use Pandas dataframe.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()

df = pd.DataFrame(data=iris.data,columns=iris.feature_names)

df["species"] = iris.target

# Optional: convert numeric species → string labels
df["species"] = df["species"].apply(lambda i: iris.target_names[i])

print(df.head())

### Visualize dataset.

Here we use the first 3 features to make a 3D scatter plot. The species (labels) are used to color the data points. We should observe that the same species are clustered together.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
%matplotlib ipympl

# Suppose df already exists with a "species" column
# Convert species names → 0,1,2
df['species_id'] = df['species'].astype('category').cat.codes
y = df['species_id'].to_numpy()

# Save list of species names in the correct numeric order
names = list(df['species'].astype('category').cat.categories)

# Convert first 3 columns to numpy
X = df.iloc[:, :3].to_numpy()
x1 = X[:, 0]
x2 = X[:, 1]
x3 = X[:, 2]

colors = ['red', 'green', 'blue']

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

for species in range(3):
    ax.scatter(
        x1[y == species],
        x2[y == species],
        x3[y == species],
        label=names[species],    # ✔ now names is defined
        color=colors[species],
        s=50
    )

ax.set_xlabel('Sepal length')
ax.set_ylabel('Sepal width')
ax.set_zlabel('Petal length')
ax.set_title('Iris Dataset (3D Visualization)')
ax.legend()

plt.show()

Load select vector machine classifier.

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

* Below we split the available data to training and test sets using `train_test_split`.
* Scale data (optional).
* Train the SVC model.
* Evaluate the model.

In [ ]:
# 1. Load the Iris dataset
iris = datasets.load_iris()
X = iris.data        # features: sepal length, sepal width, petal length, petal width
y = iris.target      # labels: 0, 1, 2 (setosa, versicolor, virginica)

# 2. Train–test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 3. (Optional but common) Feature scaling
# scaled to have a mean of 0
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 4. Define SVM classifier
# You can try different kernels: "linear", "rbf", "poly", etc.
clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)

# 5. Train the model
clf.fit(X_train_scaled, y_train)

# 6. Make predictions
y_pred = clf.predict(X_test_scaled)

# 7. Evaluate
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Confusion matrix:")
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


## Definition of precision

$$\text{Precision}= \frac{\text{True Positives}}{\text{True Positives}+\text{False Positives}}$$

It answers: “Out of everything the model said is positive, how many are truly positive?”

## Definition of Recall

$$\text{Recall}= \frac{\text{True Positives}}{\text{True Positives}+\text{False Negatives}}$$

It answers: “Out of all real positive cases, how many did the model correctly detect?

## F1 score

$$\text{F1} = 2 \frac{\text{precision}\cdot \text{recall}}{\text{precision} + \text{recall}}$$

The F1-score is a classification metric that combines precision and recall into a single number. It is the harmonic mean of precision and recall. F1 is only high when both precision and recall are high.

In [ ]:
import seaborn as sns
target_names = iris.target_names

%matplotlib 

# -------------------------
# 8. Heatmap Visualization
# -------------------------
plt.figure(figsize=(6, 5))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap (Iris Classification)")
plt.show()